In [ ]:
!pip uninstall -y transformers peft accelerate -q
!pip install -q \
  transformers==4.41.2 \
  peft==0.11.1 \
  accelerate==0.30.1 \
  datasets \
  scikit-learn \
  sentencepiece \
  safetensors

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
output_dir = "/content/drive/MyDrive/CLIP_Prompt_MFND"
os.makedirs(output_dir, exist_ok=True)
print("Save to:", output_dir)

Mounted at /content/drive
Save to: /content/drive/MyDrive/CLIP_Prompt_MFND


In [ ]:
import torch
import numpy as np
import torch.nn as nn
from datasets import load_dataset, Image
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

dataset = load_dataset("anson-huang/mirage-news")
dataset = dataset.cast_column("image", Image())
dataset = dataset.rename_column("label", "labels")

model_name = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_name)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/655M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/143M [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

In [ ]:
PROMPT_LENGTH = 5

def preprocess(examples):
    inputs = processor(
        text=examples["text"],
        images=examples["image"],
        padding="max_length",
        truncation=True,
        max_length=77 - PROMPT_LENGTH
    )
    inputs["labels"] = examples["labels"]
    return inputs

encoded_dataset = dataset.map(
    preprocess,
    remove_columns=dataset["train"].column_names,
    batched=True
)

encoded_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "pixel_values", "labels"]
)

print("Preprocessing done")

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Preprocessing done


In [ ]:
class CLIPPromptTuningForMFND(nn.Module):
    def __init__(self,
                 model_name="openai/clip-vit-base-patch32",
                 num_labels=2,
                 prompt_length=5):
        super().__init__()

        self.clip = CLIPModel.from_pretrained(model_name)
        self.prompt_length = prompt_length

        for param in self.clip.parameters():
            param.requires_grad = False

        self.hidden_dim = self.clip.config.text_config.hidden_size

        self.soft_prompt = nn.Parameter(
            torch.randn(prompt_length, self.hidden_dim)
        )

        self.classifier = nn.Linear(
            self.clip.config.projection_dim * 2,
            num_labels
        )

    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        batch_size = input_ids.size(0)
        device = input_ids.device

        token_embeddings = self.clip.text_model.embeddings.token_embedding(input_ids)

        soft_prompt = self.soft_prompt.unsqueeze(0).expand(batch_size, -1, -1)
        hidden_states = torch.cat([soft_prompt, token_embeddings], dim=1)

        prompt_mask = torch.ones(batch_size, self.prompt_length, device=device)
        attention_mask = torch.cat([prompt_mask, attention_mask], dim=1)

        seq_length = hidden_states.size(1)
        position_ids = torch.arange(seq_length, device=device).unsqueeze(0)
        position_embeddings = self.clip.text_model.embeddings.position_embedding(position_ids)
        hidden_states = hidden_states + position_embeddings

        causal_attention_mask = torch.full(
            (seq_length, seq_length),
            float("-inf"),
            device=device
        )
        causal_attention_mask = torch.triu(causal_attention_mask, diagonal=1)
        causal_attention_mask = causal_attention_mask.unsqueeze(0).unsqueeze(0)
        causal_attention_mask = causal_attention_mask.expand(batch_size, 1, seq_length, seq_length)

        if attention_mask is not None:
            attention_mask = attention_mask[:, None, None, :]
            attention_mask = attention_mask.expand(batch_size, 1, seq_length, seq_length)

        encoder_outputs = self.clip.text_model.encoder(
            inputs_embeds=hidden_states,
            attention_mask=attention_mask,
            causal_attention_mask=causal_attention_mask,
        )

        hidden_states = encoder_outputs[0]
        hidden_states = self.clip.text_model.final_layer_norm(hidden_states)

        eos_token_id = self.clip.config.text_config.eos_token_id
        eos_mask = (input_ids == eos_token_id)
        eos_positions = eos_mask.float().argmax(dim=1) + self.prompt_length

        pooled_output = hidden_states[
            torch.arange(batch_size, device=device),
            eos_positions
        ]

        text_features = self.clip.text_projection(pooled_output)

        image_outputs = self.clip.vision_model(pixel_values=pixel_values)
        image_features = self.clip.visual_projection(image_outputs.pooler_output)

        features = torch.cat([text_features, image_features], dim=1)
        logits = self.classifier(features)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss": loss, "logits": logits}

model = CLIPPromptTuningForMFND(prompt_length=PROMPT_LENGTH).to(device)

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable params: {trainable:,}")
print(f"Total params: {total:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

Trainable params: 4,610
Total params: 151,281,923
Trainable %: 0.0030%


In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    auc = roc_auc_score(labels, probs[:, 1])

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=1e-3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
trainer.train()
metrics = trainer.evaluate()
print(metrics)

trainer.save_model(output_dir)
processor.save_pretrained(output_dir)

print("✅ Model saved to:", output_dir)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
1,0.333700,0.201027,0.935600,0.930435,0.941600,0.935984,0.983417
2,0.185100,0.164529,0.944400,0.958712,0.928800,0.943519,0.988565
3,0.155300,0.143957,0.952400,0.952038,0.952800,0.952419,0.990476
4,0.127600,0.136696,0.953200,0.947119,0.960000,0.953516,0.991269
5,0.121900,0.134400,0.952800,0.948494,0.957600,0.953025,0.991515


{'eval_loss': 0.13669556379318237, 'eval_accuracy': 0.9532, 'eval_precision': 0.9471191791633781, 'eval_recall': 0.96, 'eval_f1': 0.9535160905840286, 'eval_auc': 0.9912688000000001, 'eval_runtime': 36.1718, 'eval_samples_per_second': 69.115, 'eval_steps_per_second': 2.184, 'epoch': 5.0}
✅ Model saved to: /content/drive/MyDrive/CLIP_Prompt_MFND
